# Impulse — A2D2 Bus-Signal & Camera Ingestion

Ingest one recording (`20190401_121727`) of the **A2D2 (Audi Autonomous Driving Dataset)**
into the Impulse silver-layer model, and fetch a small, **evenly-sampled-by-time** set of
front-center camera images (PNG + `cam_tstamp`) so they can be matched to the bus signals.

## ⚠️ Dataset license — read before running
A2D2 is published by **Audi AG** under **CC BY-ND 4.0** (NoDerivatives):
<https://creativecommons.org/licenses/by-nd/4.0/> · source <https://www.a2d2.audi/> ·
paper arXiv:2004.06320. Because of the **NoDerivatives** clause this repository contains
**no A2D2 data**: this notebook downloads it **at runtime, from the official source, into
your own workspace**, and writes all derived tables/images into **your own** Unity Catalog.
**Do not commit any downloaded or derived data**, and commit this notebook with outputs
cleared.

**Requirements:** Unity Catalog; serverless or DBR 14+; privilege to create a schema and a
volume; internet egress to AWS S3 (`eu-central-1`, Range requests supported); volume free
space ≈ `images_per_second × bus_window_seconds × 3.6 MB` (a few hundred MB for a demo —
*not* the full 98 GB archive).

In [ ]:
%pip install pydantic>=2.0 scipy -q
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("table_prefix", "a2d2", "Table Prefix")
dbutils.widgets.text("volume", "a2d2_raw", "UC Volume (runtime download)")
dbutils.widgets.dropdown("download_images", "true", ["true", "false"], "Download camera images")
dbutils.widgets.text("images_per_second", "0.1", "Images per second to extract from the bus window")
dbutils.widgets.text("index_scan_limit", "20000", "Camera index scan limit (0 = full walk)")
dbutils.widgets.dropdown("skip_download_if_present", "true", ["true", "false"], "Skip download if present")
dbutils.widgets.dropdown("drop_created_tables", "false", ["true", "false"], "Drop created tables")

In [ ]:
import sys, os

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")
VOLUME = dbutils.widgets.get("volume")
DOWNLOAD_IMAGES = dbutils.widgets.get("download_images") == "true"
IMAGES_PER_SECOND = float(dbutils.widgets.get("images_per_second") or "0.1")
INDEX_SCAN_LIMIT = int(dbutils.widgets.get("index_scan_limit") or "0")
SKIP_IF_PRESENT = dbutils.widgets.get("skip_download_if_present") == "true"

if not (CATALOG and SCHEMA and TABLE_PREFIX and VOLUME):
    raise ValueError("Please set catalog, schema, table_prefix and volume widgets above.")

# When run from a cloned repo / Git folder, the notebook lives at demos/a2d2/<nb> so the
# repo's src/ is 3 path-segments up; add it to sys.path so the impulse packages import.
# When deployed as a job (e.g. via the DABs bundle in this folder) the impulse wheel is
# installed instead, so we only insert src/ when it actually exists.
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = "/Workspace" + "/".join(nb_path.split("/")[:-3])
src_dir = os.path.join(REPO_ROOT, "src")
if os.path.isdir(src_dir):
    sys.path.insert(0, src_dir)

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"
CONTAINER_ID = 1
RECORDING = "20190401_121727"
BASE = "https://aev-autonomous-driving-dataset.s3.eu-central-1.amazonaws.com"
BUS_URL = f"{BASE}/camera_lidar-20190401121727_bus_signals.tar"
CAM_URL = f"{BASE}/camera_lidar-20190401121727_camera_frontcenter.tar"
LICENSE_URL = "https://creativecommons.org/licenses/by-nd/4.0/"
print(f"Target tables: {pfx}_*   (download_images={DOWNLOAD_IMAGES})")

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

vol_root = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
bus_tar_path = f"{vol_root}/bus_signals.tar"
cam_dir = f"{vol_root}/camera_lidar/{RECORDING}/camera/cam_front_center"
index_cache_path = f"{vol_root}/cam_frontcenter_index.json"
os.makedirs(cam_dir, exist_ok=True)
print("Volume root:", vol_root)

## Bus signals → silver-layer tables

Download the bus-signals tar (~185 MB), parse the 22 channels, and write the five
silver-layer tables. Channels are stored **RAW** (`container_id, channel_id, timestamp,
value`); the query engine run-length-encodes them on the fly (`data_type="RAW"`).

In [ ]:
import glob, requests

def stream_download(url, dest, chunk=8 * 1024 * 1024):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for c in r.iter_content(chunk_size=chunk):
                f.write(c)

already = SKIP_IF_PRESENT and (
    glob.glob(f"{vol_root}/**/*_bus_signals.json", recursive=True) or os.path.exists(bus_tar_path)
)
if already:
    print("Bus tar/JSON already present; skipping download.")
else:
    try:
        print("Downloading bus-signals tar (~185 MB) ...")
        stream_download(BUS_URL, bus_tar_path)
        print("Downloaded:", bus_tar_path)
    except Exception as e:
        raise RuntimeError(
            f"Could not download {BUS_URL} ({e}). If egress is blocked, download the tar "
            f"manually from https://www.a2d2.audi/ and place it at {bus_tar_path}."
        )

In [ ]:
import tarfile, glob

found = glob.glob(f"{vol_root}/**/*_bus_signals.json", recursive=True)
if SKIP_IF_PRESENT and found:
    bus_json_path = found[0]
    print("Using already-extracted bus JSON:", bus_json_path)
else:
    with tarfile.open(bus_tar_path) as t:
        member = next(m for m in t.getmembers() if m.name.endswith("_bus_signals.json"))
        t.extract(member, path=vol_root)
        bus_json_path = os.path.join(vol_root, member.name)
    print("Extracted bus JSON:", bus_json_path)

### Parse and build the silver-layer DataFrames
One **container** = this recording. Each of the 22 bus channels becomes a **channel** with
a deterministic `channel_id` (1..22, by sorted name). Per-channel statistics are computed
with the engine's own `SampleSeries` semantics for parity with query-time results.

In [ ]:
import json
import numpy as np

with open(bus_json_path) as f:
    raw = json.load(f)

channel_names = sorted(raw.keys())
assert len(channel_names) == 22, f"expected 22 channels, got {len(channel_names)}"
channel_meta = {name: {"channel_id": i + 1, "unit": raw[name].get("unit")}
                for i, name in enumerate(channel_names)}

print(f"Parsed {len(channel_names)} channels:")
for name in channel_names:
    cm = channel_meta[name]
    print(f"  [{cm['channel_id']:2d}] {name:38s} unit={str(cm['unit']):28s} n={len(raw[name]['values']):>7,}")

In [ ]:
import pandas as pd
from impulse_query_engine import schema as S

parts = []
for name in channel_names:
    arr = np.asarray(raw[name]["values"], dtype=np.float64)  # (n, 2): [ts_us, value]
    parts.append(pd.DataFrame({
        "container_id": np.int64(CONTAINER_ID),
        "channel_id": np.int32(channel_meta[name]["channel_id"]),
        "timestamp": arr[:, 0].astype(np.int64),
        "value": arr[:, 1].astype(np.float64),
    }))
channels_pdf = pd.concat(parts, ignore_index=True)
# RAW point storage; the DeltaSolver converts to RLE at query time. Do NOT dedupe here.
channels_df = spark.createDataFrame(channels_pdf, schema=S.CHANNELS_SCHEMA_WITHOUT_RLE)
print(f"channels rows: {len(channels_pdf):,}")

In [ ]:
# CHANNEL_TAGS (EAV). "channel_name" is required so query.channel(channel_name=...) resolves.
tag_rows = []
for name in channel_names:
    cid = channel_meta[name]["channel_id"]
    unit = channel_meta[name]["unit"]
    tag_rows.append((CONTAINER_ID, cid, "channel_name", name))
    tag_rows.append((CONTAINER_ID, cid, "unit", "" if unit is None else str(unit)))
channel_tags_df = spark.createDataFrame(tag_rows, schema=S.CHANNEL_TAGS)
print(f"channel_tags rows: {len(tag_rows)}")

In [ ]:
from impulse_query_engine.model.series.sample_series import SampleSeries

def weighted_std(values, w):
    wsum = w.sum()
    if wsum <= 0:
        return float("nan")
    m = np.nansum(values * w) / wsum
    return float(np.sqrt(np.nansum(w * (values - m) ** 2) / wsum))

def weighted_quantiles(values, w, qs):
    order = np.argsort(values)
    v, ww = values[order], w[order]
    cw = np.cumsum(ww)
    if cw[-1] <= 0:
        return [float("nan")] * len(qs)
    cw = cw / cw[-1]
    return [float(np.interp(q, cw, v)) for q in qs]

# CHANNEL_METRICS schema order: container_id, channel_id, value_type, sample_count,
# nan_ratio, begin_s, end_s, duration_ms, original_sample_count, original_sr,
# min, max, mean, std, pz1, pz10, pz90, pz99
metric_rows = []
for name in channel_names:
    cid = channel_meta[name]["channel_id"]
    arr = np.asarray(raw[name]["values"], dtype=np.float64)
    times, values = arr[:, 0], arr[:, 1]
    n = int(len(values))
    ss = SampleSeries.from_timestamps(times / 1e6, values)  # engine works in seconds
    dur = ss.durations()
    pz1, pz10, pz90, pz99 = weighted_quantiles(values, dur, [0.01, 0.10, 0.90, 0.99])
    metric_rows.append((
        CONTAINER_ID, cid,
        "DOUBLE",
        n,
        float(ss.nan_ratio()) if n else 0.0,
        float(times[0] / 1e6), float(times[-1] / 1e6),
        int((times[-1] - times[0]) / 1e3),
        n,
        float(np.mean(np.diff(times)) / 1e6) if n > 1 else 0.0,
        float(ss.min()), float(ss.max()), float(ss.mean()),
        weighted_std(values, dur),
        pz1, pz10, pz90, pz99,
    ))
channel_metrics_df = spark.createDataFrame(metric_rows, schema=S.CHANNEL_METRICS)
print(f"channel_metrics rows: {len(metric_rows)}")

In [ ]:
import datetime as dt
import pyspark.sql.types as T

def to_naive_utc(us):
    return dt.datetime.fromtimestamp(us / 1e6, tz=dt.timezone.utc).replace(tzinfo=None)

first = np.array([np.asarray(raw[n]["values"], dtype=np.float64)[:, 0].min() for n in channel_names])
last = np.array([np.asarray(raw[n]["values"], dtype=np.float64)[:, 0].max() for n in channel_names])
t_min, t_max = float(first.min()), float(last.max())

# Extend CONTAINER_METRICS with start_ts/stop_ts epoch-microsecond columns. These are the
# names the query engine's solver expects (SolverConfig.start_ts_col/stop_ts_col default to
# "start_ts"/"stop_ts"), so ContainerEvent and the default measurement_dimensions resolve
# them directly -- in the SAME microsecond unit as the channel timestamps (a plain rename of
# the start_dt TimestampType would cast to seconds and break alignment).
CONTAINER_METRICS_EXT = T.StructType(
    list(S.CONTAINER_METRICS.fields)
    + [T.StructField("start_ts", T.LongType()), T.StructField("stop_ts", T.LongType())]
)
container_metrics_df = spark.createDataFrame(
    [(CONTAINER_ID, to_naive_utc(t_min), to_naive_utc(t_max), int((t_max - t_min) / 1e3),
      len(channel_names), int(t_min), int(t_max))],
    schema=CONTAINER_METRICS_EXT,
)

ctags = [
    (CONTAINER_ID, "dataset", "A2D2 (Audi Autonomous Driving Dataset)"),
    (CONTAINER_ID, "recording", RECORDING),
    (CONTAINER_ID, "vehicle", "Audi"),
    (CONTAINER_ID, "source_url", BUS_URL),
    (CONTAINER_ID, "license", "CC BY-ND 4.0"),
    (CONTAINER_ID, "license_url", LICENSE_URL),
    (CONTAINER_ID, "attribution", "(c) Audi AG, A2D2, CC BY-ND 4.0"),
    (CONTAINER_ID, "camera_frontcenter_dir", cam_dir),
]
container_tags_df = spark.createDataFrame(ctags, schema=S.CONTAINER_TAGS)
print(f"recording {RECORDING}: {to_naive_utc(t_min)} .. {to_naive_utc(t_max)} UTC")

In [ ]:
writes = {
    "container_metrics": container_metrics_df,
    "container_tags": container_tags_df,
    "channel_metrics": channel_metrics_df,
    "channel_tags": channel_tags_df,
    "channels": channels_df,
}
for name, df in writes.items():
    (df.write.mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{pfx}_{name}"))
    print(f"  wrote {pfx}_{name}")
print("Bus silver-layer ingestion complete.")

## Camera images → evenly-sampled subset (PNG + timestamp)

The front-center camera tar is **98 GB** and stores frames in **random order**, but the
filename `frame_id` is monotonic with the capture time (`cam_tstamp`). We therefore:

1. **Header-walk** the tar over HTTP Range requests to index every frame's byte offsets —
   downloading *only* headers, never image payloads (compatible with small downloads).
2. **Restrict to the bus-signal time window** (the bus recording covers only ~15 min, but
   the camera tar may extend beyond it; frames outside the window have no bus data to match)
   — found by binary-searching `cam_tstamp` over the frame_id-sorted index. Then select
   **`images_per_second`** frames per second across that window (frame stride derived from
   the camera's frame rate in the index) and Range-fetch each selected frame's PNG + JSON.
3. Write the images into the volume and a small `*_camera_frames` table (with
   `cam_tstamp`) so they can be joined to the bus signals by time.

**Cost & `index_scan_limit`:** each frame's PNG and JSON are scattered, so a *complete*
(PNG+JSON) frame needs both members within the scanned range. Because storage order is
random, scanning a bounded prefix already yields frames spread across the **whole** drive's
time range — so `index_scan_limit` defaults to `20000` entries (a bounded, cached walk).
Set it to `0` to index every frame for exact whole-drive coverage (a longer one-time walk,
cached to the volume). The walk downloads only headers, never image payloads.

In [ ]:
import json as _json, re, time

FRAME_RE = re.compile(r"_(\d{9})\.(png|jso)")

def _octal(b):
    b = b.split(b"\x00")[0].strip()
    return int(b, 8) if b else 0

def _checksum_ok(h):
    stored = _octal(h[148:156])
    calc = sum(h[:148]) + sum(h[156:512]) + 32 * 8
    return stored == calc

def _read_range(session, url, off, length):
    r = session.get(url, headers={"Range": f"bytes={off}-{off + length - 1}"}, timeout=180)
    r.raise_for_status()
    return r.content

def build_index(url, scan_limit=0):
    """Walk tar headers (no payload downloads). Returns {frame_id: {'png':{off,size,name},
    'json':{off,size}}}. Handles GNU @LongLink (typeflag 'L') and directory entries."""
    frames, off, entries = {}, 0, 0
    long_named = False
    t0 = time.time()
    with requests.Session() as s:
        while True:
            h = _read_range(s, url, off, 512)
            if len(h) < 512 or h == b"\x00" * 512:
                break
            if not _checksum_ok(h):
                raise RuntimeError(f"tar header checksum mismatch at offset {off}; walk aborted")
            typ = h[156:157]
            size = _octal(h[124:136])
            data_off = off + 512
            if typ == b"L":
                # Skip the long-name data block; the next file header's (truncated) name
                # still carries the frame_id, so we don't need to read the name here.
                off = data_off + ((size + 511) // 512) * 512
                long_named = True
                continue
            long_named = False
            if typ != b"5":  # not a directory
                name = h[0:100].split(b"\x00")[0].decode("latin1").split("/")[-1]
                m = FRAME_RE.search(name)
                if m:
                    fid = int(m.group(1))
                    rec = frames.setdefault(fid, {})
                    if ".png" in name:
                        rec["png"] = {"off": data_off, "size": size, "name": name}
                    elif ".jso" in name:
                        rec["json"] = {"off": data_off, "size": size}
                entries += 1
                if entries % 5000 == 0:
                    print(f"  ...indexed {entries:,} entries, {len(frames):,} frames "
                          f"({time.time() - t0:.0f}s)")
                if scan_limit and entries >= scan_limit:
                    break
            off = data_off + ((size + 511) // 512) * 512
    return frames

frame_index = {}
if DOWNLOAD_IMAGES:
    if SKIP_IF_PRESENT and os.path.exists(index_cache_path):
        with open(index_cache_path) as f:
            frame_index = {int(k): v for k, v in _json.load(f).items()}
        print(f"Loaded cached camera index: {len(frame_index):,} frames")
    else:
        print(f"Walking camera-tar headers (scan_limit={INDEX_SCAN_LIMIT or 'full'}); "
              f"this downloads only headers, no images ...")
        idx = build_index(CAM_URL, scan_limit=INDEX_SCAN_LIMIT)
        frame_index = {k: v for k, v in idx.items() if "png" in v and "json" in v}
        with open(index_cache_path, "w") as f:
            _json.dump(frame_index, f)
        print(f"Indexed {len(frame_index):,} complete frames; cached to {index_cache_path}")
else:
    print("download_images=false -> skipping camera image fetch.")

In [ ]:
import pyspark.sql.types as T
import datetime as dt

CAMERA_FRAMES_SCHEMA = T.StructType([
    T.StructField("container_id", T.LongType(), False),
    T.StructField("frame_id", T.IntegerType(), False),
    T.StructField("cam_tstamp_us", T.LongType()),
    T.StructField("cam_dt", T.TimestampType()),
    T.StructField("png_path", T.StringType()),
])

if DOWNLOAD_IMAGES and frame_index:
    sorted_ids = sorted(frame_index.keys())          # frame_id ascending == time order
    os.makedirs(cam_dir, exist_ok=True)
    rows, bytes_fetched = [], 0
    ts_cache = {}
    with requests.Session() as s:
        def cam_ts(fid):
            if fid not in ts_cache:
                js = frame_index[fid]["json"]
                jb = _read_range(s, CAM_URL, js["off"], js["size"])
                ts_cache[fid] = int(_json.loads(jb.decode("latin1"))["cam_tstamp"])
            return ts_cache[fid]

        # Restrict to the bus-signal time window [t_min, t_max] (microseconds, from the bus
        # ingestion above): frames outside it have no bus data to correlate with, so there's
        # no point downloading them. cam_tstamp is monotonic in frame_id, so binary-search
        # the window bounds -- only a handful of JSON header reads, no image payloads.
        def lower_bound(ids, target):
            a, b = 0, len(ids)
            while a < b:
                m = (a + b) // 2
                if cam_ts(ids[m]) < target: a = m + 1
                else: b = m
            return a
        def upper_bound(ids, target):
            a, b = 0, len(ids)
            while a < b:
                m = (a + b) // 2
                if cam_ts(ids[m]) <= target: a = m + 1
                else: b = m
            return a

        lo, hi = lower_bound(sorted_ids, t_min), upper_bound(sorted_ids, t_max)
        window_ids = sorted_ids[lo:hi]
        if not window_ids:
            raise RuntimeError("No camera frames fall inside the bus time window.")
        # Select ~IMAGES_PER_SECOND frames per second across the window. cam_tstamp is
        # ~uniform in frame_id, so derive a frame stride from the frame rate available in the
        # index (cheap: uses only the two window-boundary timestamps, already cached above).
        t_lo, t_hi = cam_ts(window_ids[0]), cam_ts(window_ids[-1])
        dur_s = max((t_hi - t_lo) / 1e6, 1e-9)
        avail_fps = max(len(window_ids) - 1, 1) / dur_s
        stride = max(1, round(avail_fps / IMAGES_PER_SECOND))
        selected = window_ids[::stride]
        eff_rate = (len(selected) - 1) / dur_s if len(selected) > 1 else 0.0
        print(f"Indexed F={len(sorted_ids):,} frames; {len(window_ids):,} fall inside the bus "
              f"window {to_naive_utc(t_min)} .. {to_naive_utc(t_max)} UTC ({dur_s:.0f}s). "
              f"Requested {IMAGES_PER_SECOND} img/s -> frame stride {stride} -> {len(selected)} "
              f"images (~{eff_rate:.3f} img/s).")

        for i, fid in enumerate(selected):
            rec = frame_index[fid]
            png = rec["png"]
            png_path = f"{cam_dir}/{png['name']}"
            cam_tstamp = cam_ts(fid); bytes_fetched += rec["json"]["size"]
            if not (SKIP_IF_PRESENT and os.path.exists(png_path)):
                pb = _read_range(s, CAM_URL, png["off"], png["size"]); bytes_fetched += len(pb)
                with open(png_path, "wb") as f:
                    f.write(pb)
            cam_dt = dt.datetime.fromtimestamp(cam_tstamp / 1e6, tz=dt.timezone.utc).replace(tzinfo=None)
            rows.append((CONTAINER_ID, int(fid), cam_tstamp, cam_dt, png_path))
            if (i + 1) % 25 == 0:
                print(f"  fetched {i + 1}/{len(selected)} frames, {bytes_fetched / 1e6:.1f} MB")
    camera_frames_df = spark.createDataFrame(rows, schema=CAMERA_FRAMES_SCHEMA)
    (camera_frames_df.write.mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{pfx}_camera_frames"))
    print(f"Wrote {len(rows)} rows to {pfx}_camera_frames; images in {cam_dir}; "
          f"total bytes fetched: {bytes_fetched / 1e6:.1f} MB")
else:
    print("No images fetched (download_images=false or empty index).")

## Validation — TSAL round-trip and image↔bus time-join

In [ ]:
from databricks.sdk import WorkspaceClient
from impulse_reporting.core.report import Report
from impulse_reporting.core.page import Page
from impulse_reporting.events.basic_event import BasicEvent
from impulse_reporting.aggregations.histogram import HistogramDuration
import pyspark.sql.functions as F

config = {
    "source": {
        "container_metrics_table": f"{pfx}_container_metrics",
        "container_tags_table": f"{pfx}_container_tags",
        "channel_metrics_table": f"{pfx}_channel_metrics",
        "channel_tags_table": f"{pfx}_channel_tags",
        "channels_uri": f"{pfx}_channels",
    },
    "unity_sink": {"catalog": CATALOG, "schema": SCHEMA, "table_prefix": TABLE_PREFIX},
    "query_engine": {"solver": "DeltaSolver", "data_type": "RAW"},
    "measurement_dimensions": ["container_id"],
}
report = Report(name="a2d2_validation", spark=spark, workspace_client=WorkspaceClient(), config=config)
db = report.get_db()

# Validation (same pattern as getting_started.ipynb): a 1D duration histogram of
# vehicle speed, scoped to a "vehicle moving" event, persisted to the gold star schema.
veh_speed = db.query.channel(channel_name="vehicle_speed")

moving = BasicEvent(name="vehicle_moving", expr=veh_speed > 0, desc="vehicle speed > 0 km/h")
report.add_event(moving)

page = Page(page_number=1)
page.add_aggregation(
    HistogramDuration(
        name="vehicle_speed_distribution",
        base_expr=veh_speed,
        bins=[float(x) for x in range(0, 161, 10)],
        event=moving,
        channel_name="vehicle_speed",
        bins_unit="km/h",
        values_unit="s",
    )
)
report.add_page(page)

report.determine_report()
report.persist_results()

print("Gold tables written. Duration-weighted vehicle-speed distribution:")
display(
    spark.read.table(f"{pfx}_histogram_fact")
    .groupBy("bin_name", "lower_bound")
    .agg(F.sum("hist_value").alias("duration_us"))
    .orderBy("lower_bound")
)

In [ ]:
# Match sampled camera frames to bus data by nearest timestamp (uses cam_tstamp).
if DOWNLOAD_IMAGES and spark.catalog.tableExists(f"{pfx}_camera_frames"):
    sp_cid = (spark.read.table(f"{pfx}_channel_tags")
              .where("key = 'channel_name' AND value = 'vehicle_speed'")
              .select("channel_id").first()[0])
    sp = (spark.read.table(f"{pfx}_channels").where(f"channel_id = {sp_cid}")
          .select("timestamp", "value").orderBy("timestamp").toPandas())
    cf = spark.read.table(f"{pfx}_camera_frames").orderBy("cam_tstamp_us").toPandas()

    idx = np.clip(np.searchsorted(sp["timestamp"].values, cf["cam_tstamp_us"].values), 0, len(sp) - 1)
    cf["vehicle_speed_kmh"] = sp["value"].values[idx]
    display(spark.createDataFrame(cf[["frame_id", "cam_dt", "vehicle_speed_kmh", "png_path"]]))

    # Render one sample frame (best-effort).
    try:
        import matplotlib.pyplot as plt
        import matplotlib.image as mpimg
        p = cf["png_path"].iloc[len(cf) // 2]
        plt.figure(figsize=(10, 6)); plt.imshow(mpimg.imread(p)); plt.axis("off")
        plt.title(f"frame {cf['frame_id'].iloc[len(cf) // 2]} @ {cf['cam_dt'].iloc[len(cf) // 2]} UTC")
        plt.show()
    except Exception as e:
        print("image preview skipped:", e)
else:
    print("No camera_frames table to join (images not downloaded).")

## Cleanup (optional)
Set the **drop_created_tables** widget to `true` and run the next cell to remove the tables
created by this notebook. Extracted images on the volume are left in place.

In [ ]:
if dbutils.widgets.get("drop_created_tables") == "true":
    # Drops everything this notebook created: silver tables, the gold star-schema tables
    # written by persist_results(), and camera_frames -- i.e. all {TABLE_PREFIX}_* tables.
    tbls = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA} LIKE '{TABLE_PREFIX}_*'").collect()
    for r in tbls:
        spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.{r['tableName']}")
        print(f"  dropped {CATALOG}.{SCHEMA}.{r['tableName']}")
    print(f"Dropped {len(tbls)} {TABLE_PREFIX}_* tables. "
          f"(Volume + extracted images left in place; remove manually if desired.)")
else:
    print("drop_created_tables=false -> nothing dropped.")